In [1]:
!pip install h2o

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 MB 5.0 MB/s eta 0:00:00


In [19]:
import pandas as pd
import numpy as np

In [20]:
df_bat = pd.read_csv('/content/ipl_bat_features_flagv1.csv')
df_bowl = pd.read_csv('/content/ipl_bowl_features_flagv1.csv')
df_bat_fp = pd.read_csv('/content/ipl_bat_fp.csv')
df_bowl_fp = pd.read_csv('/content/ipl_bowl_fp.csv')

In [21]:
import pandas as pd

# Load the CSV file
def add_season_column(input_file, output_file):
    # Read the CSV file
    df = pd.read_csv(input_file)

    # Ensure 'date' column is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Extract the year from the 'date' column and create a new 'season' column
    df['season'] = df['date'].dt.year

    # Save the modified DataFrame to a new CSV file
    df.to_csv(output_file, index=False)

# Example usage
input_csv = '/content/ipl_bat_features_flagv1.csv'  # replace with your input file path
output_csv = '/content/ipl_bat_features_flagv1.csv'  # replace with your desired output file path
add_season_column(input_csv, output_csv)



In [22]:
df_bat=pd.read_csv('/content/ipl_bat_features_flagv1.csv')
df_bat.shape,df_bowl.shape

((24333, 38), (24444, 38))

# DATA PREPARATION

In [23]:
df_bowl['date'] = pd.to_datetime(df_bowl['date'])
df_bat['date'] = pd.to_datetime(df_bat['date'])

In [24]:
# merging fantasy points in one single dataset

df_bat = df_bat.merge(
    df_bat_fp[['batting_fp', 'match_id', 'batter_id']],
    on = ['match_id', 'batter_id']
)

df_bowl = df_bowl.merge(
    df_bowl_fp[['bowling_fp', 'match_id', 'bowler_id']],
    on = ['match_id', 'bowler_id']
)

In [25]:
#sorting values by date and innings
df_bat = df_bat.sort_values(by = ['date', 'inns'])
df_bowl = df_bowl.sort_values(by = ['date', 'inns'])
df_bat.head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,venue,career_avg,career_rpm,...,last5_sr,last5_rpm,last5_bfpm,last5_bpm,career_debut,season_debut,venue_debut,innings_debut,season,batting_fp
929,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,26184,Ajit Agarkar,M Chinnaswamy Stadium,0.0,0.0,...,0.0,0.0,0.0,0.0,1,1,1,1,2008,0
2340,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,227712,Ashok Dinda,M Chinnaswamy Stadium,0.0,0.0,...,0.0,0.0,0.0,0.0,1,1,1,1,2008,0
3295,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,37737,Brendon McCullum,M Chinnaswamy Stadium,0.0,0.0,...,0.0,0.0,0.0,0.0,1,1,1,1,2008,298
4336,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,5766,David Hussey,M Chinnaswamy Stadium,0.0,0.0,...,0.0,0.0,0.0,0.0,1,1,1,1,2008,16
7649,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,236779,Ishant Sharma,M Chinnaswamy Stadium,0.0,0.0,...,0.0,0.0,0.0,0.0,1,1,1,1,2008,0


In [26]:
df_bowl.head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,venue,season,career_avg,...,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm,career_debut,season_debut,venue_debut,innings_debut,bowling_fp
2384,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,6938,Ashley Noffke,1,M Chinnaswamy Stadium,2008,0.0,...,0.0,0.0,0.0,0.0,0.0,1,1,1,1,31
2811,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,26206,Balachandra Akhil,1,M Chinnaswamy Stadium,2008,0.0,...,0.0,0.0,0.0,0.0,0.0,1,1,1,1,0
3548,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,8291,Cameron White,1,M Chinnaswamy Stadium,2008,0.0,...,0.0,0.0,0.0,0.0,0.0,1,1,1,1,1
7924,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,45789,Jacques Kallis,1,M Chinnaswamy Stadium,2008,0.0,...,0.0,0.0,0.0,0.0,0.0,1,1,1,1,29
12047,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,44111,Mark Boucher,1,M Chinnaswamy Stadium,2008,0.0,...,0.0,0.0,0.0,0.0,0.0,1,1,1,1,0


In [27]:
# Label Encoding categorical features

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
for column in ['bat_team', 'bowl_team', 'venue']:
    df_bat[column] = encoder.fit_transform(df_bat[column])
for column in ['bat_team', 'bowl_team', 'venue']:
    df_bowl[column] = encoder.fit_transform(df_bowl[column])

In [28]:
# dropping columns not required for training
df_bat.drop(columns=['date', 'match_id', 'batter'], inplace=True)
df_bowl.drop(columns=['date', 'match_id', 'bowler'], inplace=True)

In [30]:
df_bat.shape, df_bowl.shape

((24333, 36), (24664, 36))

# TRAIN TEST SPLIT

In [31]:
# TRAIN VAL TEST SPLIT

df_bat_train = df_bat[df_bat['season'] < 2023]
df_bat_val = df_bat[df_bat['season'] >= 2023]
df_bowl_train = df_bowl[df_bowl['season'] < 2023]
df_bowl_val = df_bowl[df_bowl['season'] >= 2023]

In [32]:
df_bat_train.shape, df_bat_val.shape,df_bowl_train.shape, df_bowl_val.shape

((20878, 36), (3455, 36), (21208, 36), (3456, 36))

# BATTING MODEL TRAINING


In [34]:
import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import numpy as np

In [35]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.26" 2025-01-21; OpenJDK Runtime Environment (build 11.0.26+4-post-Ubuntu-1ubuntu122.04); OpenJDK 64-Bit Server VM (build 11.0.26+4-post-Ubuntu-1ubuntu122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.11/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpe6kn58fe
  JVM stdout: /tmp/tmpe6kn58fe/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpe6kn58fe/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,05 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,6 days
H2O_cluster_name:,H2O_from_python_unknownUser_bel1kr
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.170 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [36]:
# Convert to H2OFrame
train = h2o.H2OFrame(df_bat_train)
valid = h2o.H2OFrame(df_bat_val)

# Define features and target variable
target = "batting_fp"   # Replace with your actual target column name
features = [col for col in train.columns if col != target]

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [37]:
# Run AutoML for 60 minutes
aml = H2OAutoML(
    max_runtime_secs=3600,   # 1 hour
    stopping_metric="RMSE",  # Since it's a regression problem
    seed=42
)
aml.train(x=features, y=target, training_frame=train, validation_frame=valid)

AutoML progress: |
20:33:32.152: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.

███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),5/6
# GBM base models (used / total),1/1
# XGBoost base models (used / total),1/1
# DeepLearning base models (used / total),1/1
# GLM base models (used / total),1/1
# DRF base models (used / total),1/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [38]:
lb = aml.leaderboard
lb

model_id,rmse,mse,mae,rmsle,mean_residual_deviance
StackedEnsemble_BestOfFamily_4_AutoML_1_20250402_203332,31.7201,1006.17,20.8511,nan,1006.17
StackedEnsemble_BestOfFamily_6_AutoML_1_20250402_203332,31.7287,1006.71,20.8546,nan,1006.71
StackedEnsemble_AllModels_3_AutoML_1_20250402_203332,31.7453,1007.76,20.8789,nan,1007.76
StackedEnsemble_AllModels_2_AutoML_1_20250402_203332,31.7462,1007.82,20.8861,nan,1007.82
StackedEnsemble_BestOfFamily_3_AutoML_1_20250402_203332,31.7499,1008.05,20.8928,nan,1008.05
StackedEnsemble_AllModels_4_AutoML_1_20250402_203332,31.7509,1008.12,20.8789,nan,1008.12
StackedEnsemble_AllModels_1_AutoML_1_20250402_203332,31.7652,1009.03,20.8869,nan,1009.03
StackedEnsemble_BestOfFamily_2_AutoML_1_20250402_203332,31.772,1009.46,20.8992,nan,1009.46
StackedEnsemble_BestOfFamily_1_AutoML_1_20250402_203332,31.7762,1009.73,20.8866,nan,1009.73
XGBoost_lr_search_selection_AutoML_1_20250402_203332_select_grid_model_2,31.8149,1012.19,20.9,nan,1012.19


In [39]:
best_model = aml.leader  # Best model
performance = best_model.model_performance(valid)
print(performance)

ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 1209.7891279890864
RMSE: 34.78202305773898
MAE: 22.34048622693525
RMSLE: NaN
Mean Residual Deviance: 1209.7891279890864
R^2: 0.25998366539946893
Null degrees of freedom: 3454
Residual degrees of freedom: 3449
Null deviance: 5669707.686377454
Residual deviance: 4179821.437202294
AIC: 34343.15092528785


# BOWLING MODEL TRAINING

In [40]:
df_bowl_fp.shape,df_bowl.shape

((24444, 4), (24664, 36))

In [41]:
# Convert to H2OFrame
train = h2o.H2OFrame(df_bowl_train)
valid = h2o.H2OFrame(df_bowl_val)

# Define features and target variable
target = "bowling_fp"   # Replace with your actual target column name
features = [col for col in train.columns if col != target]

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [42]:
# Run AutoML for 40 minutes
aml = H2OAutoML(
    max_runtime_secs=2500,   # 2500 seconds
    stopping_metric="RMSE",  # Since it's a regression problem
    seed=42
)
aml.train(x=features, y=target, training_frame=train, validation_frame=valid)

AutoML progress: |
21:38:51.295: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.

███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),6/6
# GBM base models (used / total),1/1
# XGBoost base models (used / total),1/1
# GLM base models (used / total),1/1
# DeepLearning base models (used / total),1/1
# DRF base models (used / total),2/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [44]:
# checking top performers

lb = aml.leaderboard
lb

model_id,rmse,mse,mae,rmsle,mean_residual_deviance
StackedEnsemble_BestOfFamily_4_AutoML_2_20250402_213851,23.1223,534.64,13.9854,nan,534.64
StackedEnsemble_BestOfFamily_3_AutoML_2_20250402_213851,23.1386,535.394,14.0227,nan,535.394
StackedEnsemble_AllModels_3_AutoML_2_20250402_213851,23.1388,535.404,14.0173,nan,535.404
StackedEnsemble_AllModels_4_AutoML_2_20250402_213851,23.1436,535.628,13.9677,nan,535.628
StackedEnsemble_AllModels_2_AutoML_2_20250402_213851,23.1466,535.764,14.021,nan,535.764
StackedEnsemble_BestOfFamily_2_AutoML_2_20250402_213851,23.15,535.923,14.0629,nan,535.923
StackedEnsemble_AllModels_1_AutoML_2_20250402_213851,23.1527,536.048,14.0505,nan,536.048
GLM_1_AutoML_2_20250402_213851,23.1553,536.168,14.1343,nan,536.168
StackedEnsemble_BestOfFamily_1_AutoML_2_20250402_213851,23.1576,536.274,14.0686,nan,536.274
XGBoost_grid_1_AutoML_2_20250402_213851_model_4,23.2038,538.418,13.9878,nan,538.418


In [46]:
best_model = aml.leader  # Best model
performance = best_model.model_performance(valid)
print(performance)

ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 501.0658143491169
RMSE: 22.384499421454947
MAE: 13.127076033432536
RMSLE: NaN
Mean Residual Deviance: 501.0658143491169
R^2: 0.3453165311667724
Null degrees of freedom: 3455
Residual degrees of freedom: 3449
Null deviance: 2652086.1158760176
Residual deviance: 1731683.4543905482
AIC: 31308.747797822012
